# Update a policy

`PUT /admin/policies/{id}` rewrites the file. The harness version rises only
when the bytes or the title change, and the connector writes one new S3
version. The same body again changes nothing, so the version count stays.


In [1]:
import sys
from pathlib import Path

here = Path.cwd()
if not (here / "hr_demo.py").exists():
    here = here / "notebooks"
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

import hr_demo


## Precondition and create


In [2]:
POLICY_ID = "pol-demo-update"
KEY = f"raw/hr/policy/{POLICY_ID}.md"
BODY = "# Demo policy\n\nCreated by the notebook.\n"
PAYLOAD = {
    "policy_id": POLICY_ID,
    "title": "Demo policy",
    "category": "leave",
    "region": "DE",
    "format": "markdown",
    "native_acl": ["hr_employee"],
    "body": BODY,
}
hr_demo.delete_if_present("policy", POLICY_ID)
created = hr_demo.admin("POST", "/admin/policies", PAYLOAD)
print(created["policy_id"], "version", created["version"], created["native_acl"])


pol-demo-update version 1 ['hr_employee']


In [3]:
def ready() -> bool:
    head = hr_demo.snapshot(KEY)
    if head is None:
        return False
    meta = head["Metadata"]
    return (
        BODY.encode() in hr_demo.object_body(KEY)
        and meta.get(hr_demo.META_ACL) == "hr_employee"
        and meta.get(hr_demo.META_VERSION) == str(created["version"])
        and len(meta.get(hr_demo.META_HASH, "")) == 64
        and all(meta.get(name) for name in hr_demo.META_KEYS)
    )

hr_demo.wait_until(ready, hr_demo.WRITE_WAIT, "create snapshot")
print(hr_demo.snapshot(KEY)["Metadata"])


{'kbs-content-hash': '4166869738e975d3f2747dac7ca2a132bd0345f3e6a30c2314bdde19821d5b9d', 'kbs-native-acl': 'hr_employee', 'kbs-snapshot-at': '2026-10-16T16:52:17Z', 'kbs-source-updated-at': '2026-10-16T16:52:14.119299Z', 'kbs-version': '1'}


## Edit


In [4]:
EDITED = "# Demo policy\n\nEdited by the notebook.\n"
before_versions = hr_demo.versions(KEY)
edited = hr_demo.admin("PUT", f"/admin/policies/{POLICY_ID}", {"body": EDITED})
assert edited["version"] == created["version"] + 1

def rewritten() -> bool:
    head = hr_demo.snapshot(KEY)
    if head is None:
        return False
    meta = head["Metadata"]
    return (
        hr_demo.versions(KEY) == before_versions + 1
        and meta.get(hr_demo.META_VERSION) == str(edited["version"])
        and EDITED.encode() in hr_demo.object_body(KEY)
    )

hr_demo.wait_until(rewritten, hr_demo.WRITE_WAIT, "policy rewrite")
print("versions", hr_demo.versions(KEY), "metadata", hr_demo.snapshot(KEY)["Metadata"])


versions 2 metadata {'kbs-content-hash': '3d2ef63496f9db6575ad251a4a4556ce031d3240fbeac5000bb7d94841289a22', 'kbs-native-acl': 'hr_employee', 'kbs-snapshot-at': '2026-10-16T16:52:27Z', 'kbs-source-updated-at': '2026-10-16T16:52:18.388447Z', 'kbs-version': '2'}


## Same edit again


In [5]:
stable_versions = hr_demo.versions(KEY)
stable = hr_demo.admin("GET", f"/policies/{POLICY_ID}")
again = hr_demo.admin("PUT", f"/admin/policies/{POLICY_ID}", {"body": EDITED})
assert again["version"] == stable["version"]
hr_demo.wait_cycles(1)
assert hr_demo.versions(KEY) == stable_versions
print("unchanged versions", stable_versions)


unchanged versions 2


## Negative case


In [6]:
missing = hr_demo.expect_status(
    "PUT",
    "/admin/policies/pol-demo-missing",
    404,
    {"body": "# Missing\n"},
)
print("missing", missing.status_code)


missing 404


## Cleanup


In [7]:
hr_demo.delete_if_present("policy", POLICY_ID)
hr_demo.wait_until(
    lambda: hr_demo.versions(KEY) == 0,
    hr_demo.DELETE_WAIT,
    "cleanup delete",
)
print("removed", KEY)


removed raw/hr/policy/pol-demo-update.md
